In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from random import randint

In [2]:
url = "https://www.budgetbytes.com/category/recipes/main-dish/"

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
response.status_code # Check if the request was successful

200

In [4]:
# If the request was successful, parse the content
soup = BeautifulSoup(response.text, "html.parser")

In [5]:
# Find all recipe tiles on the page
recipe_tiles = soup.find_all('article', class_ = "post-summary")
len(recipe_tiles)

12

In [6]:
# Function to extract prep time, cook time, and servings from a recipe page
def get_info(link):
    response = requests.get(link, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")

    prep = soup.find("span", class_="wprm-recipe-prep_time-minutes")
    cook = soup.find("span", class_="wprm-recipe-cook_time-minutes")
    serv = soup.find("span", class_="wprm-recipe-servings")

    # Extract text if the elements are found, otherwise return None
    prep_time = prep.text if prep else None
    cook_time = cook.text if cook else None
    servings = serv.text if serv else None

    return prep_time, cook_time, servings 

In [7]:
# Create an empty list to store the data
rows = []

In [8]:
# Loop through each recipe tile and extract the required information
# This is only the first page, we will need to loop through multiple pages to get more data
for recipe in recipe_tiles:
    find_title = recipe.find('h3', class_ = "post-summary__title")
    find_cost = recipe.find('span', class_ = "cost-per")
    find_link = recipe.find('a')

    if find_title and find_cost and find_link:
        link = find_link['href']
        prep_time, cook_time, servings = get_info(link)

        rows.append({
            "title": find_title.text,
            "cost": find_cost.text,
            "prep_time": prep_time,
            "cook_time": cook_time,
            "servings": servings
        })  
    
    time.sleep(1)  # Sleep for 1 second to be polite to the server

# Check how many recipes we have collected
len(rows)

12

In [9]:
# Now we will loop through multiple pages to get more data
for page in range(2, 31):  # Scrape pages 2 to 30

    url = f"https://www.budgetbytes.com/category/recipes/main-dish/page/{page}/"

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html")

    recipe_tiles = soup.find_all('article', class_="post-summary")

    for recipe in recipe_tiles:
        find_title = recipe.find('h3', class_ = "post-summary__title")
        find_cost = recipe.find('span', class_ = "cost-per")
        find_link = recipe.find('a')

        if find_title and find_cost and find_link:
            link = find_link['href']
            prep_time, cook_time, servings = get_info(link)

            rows.append({
                "title": find_title.text,
                "cost": find_cost.text,
                "prep_time": prep_time,
                "cook_time": cook_time,
                "servings": servings
            })  
        time.sleep(randint(1, 3))  # Sleep for 1-3 seconds to be polite to the server
    time.sleep(randint(1, 3))  # Sleep for 1-3 seconds to be polite to the server

In [10]:
# Check how many recipes we have collected
len(rows)

357

In [11]:
# Create a DataFrame from the collected data
recipes = pd.DataFrame(rows)

In [12]:
# Preview the data
recipes

,title,cost,prep_time,cook_time,servings
0,Sliders,$11.44 recipe / $1.91 serving,10 minutes,25 minutes,6
1,Ham Tetrazzini,$7.65 recipe / $1.28 serving,15 minutes,45 minutes,6
2,Chicken Broth Bowl,$7.53 recipe / $1.89 serving,15 minutes,35 minutes,4
3,Feta Pasta,$11.05 recipe / $2.76 serving,20 minutes,40 minutes,4
4,Eggplant Curry,$7.32 recipe / $1.83 serving,15 minutes,45 minutes,4
...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,$3.69 recipe / $0.92 serving,15 minutes,25 minutes,4
353,BBQ Bean Sliders,$5.85 recipe / $1.46 serving,15 minutes,15 minutes,4
354,Smoky Roasted Sausage and Vegetables,$7.87 recipe / $1.97 serving,15 minutes,40 minutes,4
355,One Pot Spinach and Artichoke Pasta,$9.07 recipe / $1.62 serving,10 minutes,15 minutes,6


In [13]:
# Keep only unique recipes
recipes = recipes.drop_duplicates(subset='title')

In [14]:
# Reset the index after dropping duplicates
recipes.reset_index(drop=True, inplace=True)
recipes

,title,cost,prep_time,cook_time,servings
0,Sliders,$11.44 recipe / $1.91 serving,10 minutes,25 minutes,6
1,Ham Tetrazzini,$7.65 recipe / $1.28 serving,15 minutes,45 minutes,6
2,Chicken Broth Bowl,$7.53 recipe / $1.89 serving,15 minutes,35 minutes,4
3,Feta Pasta,$11.05 recipe / $2.76 serving,20 minutes,40 minutes,4
4,Eggplant Curry,$7.32 recipe / $1.83 serving,15 minutes,45 minutes,4
...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,$3.69 recipe / $0.92 serving,15 minutes,25 minutes,4
353,BBQ Bean Sliders,$5.85 recipe / $1.46 serving,15 minutes,15 minutes,4
354,Smoky Roasted Sausage and Vegetables,$7.87 recipe / $1.97 serving,15 minutes,40 minutes,4
355,One Pot Spinach and Artichoke Pasta,$9.07 recipe / $1.62 serving,10 minutes,15 minutes,6


In [15]:
# Split cost into total cost and cost per serving
recipes[['cost', 'cost_per_serving']] = recipes['cost'].str.split(' / ', expand=True)

In [16]:
recipes

,title,cost,prep_time,cook_time,servings,cost_per_serving
0,Sliders,$11.44 recipe,10 minutes,25 minutes,6,$1.91 serving
1,Ham Tetrazzini,$7.65 recipe,15 minutes,45 minutes,6,$1.28 serving
2,Chicken Broth Bowl,$7.53 recipe,15 minutes,35 minutes,4,$1.89 serving
3,Feta Pasta,$11.05 recipe,20 minutes,40 minutes,4,$2.76 serving
4,Eggplant Curry,$7.32 recipe,15 minutes,45 minutes,4,$1.83 serving
...,...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,$3.69 recipe,15 minutes,25 minutes,4,$0.92 serving
353,BBQ Bean Sliders,$5.85 recipe,15 minutes,15 minutes,4,$1.46 serving
354,Smoky Roasted Sausage and Vegetables,$7.87 recipe,15 minutes,40 minutes,4,$1.97 serving
355,One Pot Spinach and Artichoke Pasta,$9.07 recipe,10 minutes,15 minutes,6,$1.62 serving


In [17]:
# Fill missing cook_time and prep_time values with "0"
recipes['cook_time'] = recipes['cook_time'].fillna("0")
recipes['prep_time'] = recipes['prep_time'].fillna("0")

In [18]:
# Convert cost, cost_per_serving, prep_time, and cook_time to numeric values
recipes['cost'] = recipes['cost'].str.extract(r'(\d+\.\d+)').astype(float)
recipes['cost_per_serving'] = recipes['cost_per_serving'].str.extract(r'(\d+\.\d+)').astype(float)
recipes['prep_time'] = recipes['prep_time'].str.extract(r'(\d+)').astype(int)
recipes['cook_time'] = recipes['cook_time'].str.extract(r'(\d+)').astype(int)

In [19]:
recipes

,title,cost,prep_time,cook_time,servings,cost_per_serving
0,Sliders,11.44,10,25,6,1.91
1,Ham Tetrazzini,7.65,15,45,6,1.28
2,Chicken Broth Bowl,7.53,15,35,4,1.89
3,Feta Pasta,11.05,20,40,4,2.76
4,Eggplant Curry,7.32,15,45,4,1.83
...,...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,3.69,15,25,4,0.92
353,BBQ Bean Sliders,5.85,15,15,4,1.46
354,Smoky Roasted Sausage and Vegetables,7.87,15,40,4,1.97
355,One Pot Spinach and Artichoke Pasta,9.07,10,15,6,1.62


In [20]:
# Create a new column for total time by adding prep_time and cook_time
recipes['total_time'] = recipes['prep_time'] + recipes['cook_time']

In [21]:
recipes

,title,cost,prep_time,cook_time,servings,cost_per_serving,total_time
0,Sliders,11.44,10,25,6,1.91,35
1,Ham Tetrazzini,7.65,15,45,6,1.28,60
2,Chicken Broth Bowl,7.53,15,35,4,1.89,50
3,Feta Pasta,11.05,20,40,4,2.76,60
4,Eggplant Curry,7.32,15,45,4,1.83,60
...,...,...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,3.69,15,25,4,0.92,40
353,BBQ Bean Sliders,5.85,15,15,4,1.46,30
354,Smoky Roasted Sausage and Vegetables,7.87,15,40,4,1.97,55
355,One Pot Spinach and Artichoke Pasta,9.07,10,15,6,1.62,25


In [22]:
# Reorder the columns to have a cleaner look
recipes = recipes[
    ['title','cost', 'cost_per_serving', 'servings', 'prep_time', 'cook_time', 'total_time']
]

In [23]:
recipes

,title,cost,cost_per_serving,servings,prep_time,cook_time,total_time
0,Sliders,11.44,1.91,6,10,25,35
1,Ham Tetrazzini,7.65,1.28,6,15,45,60
2,Chicken Broth Bowl,7.53,1.89,4,15,35,50
3,Feta Pasta,11.05,2.76,4,20,40,60
4,Eggplant Curry,7.32,1.83,4,15,45,60
...,...,...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,3.69,0.92,4,15,25,40
353,BBQ Bean Sliders,5.85,1.46,4,15,15,30
354,Smoky Roasted Sausage and Vegetables,7.87,1.97,4,15,40,55
355,One Pot Spinach and Artichoke Pasta,9.07,1.62,6,10,15,25


In [53]:
recipes.to_csv("budgetbytes_recipes.csv", index=False)

In [ ]:
recipes[['cost', 'cost_per_serving', 'total_time']].describe()

,cost,cost_per_serving,total_time
count,357.000000,354.000000,357.000000
mean,9.369255,1.901695,38.187675
std,3.852566,0.766514,15.862686
min,1.510000,0.270000,0.000000
25%,6.780000,1.390000,25.000000
50%,8.824000,1.835000,35.000000
75%,11.250000,2.307500,50.000000
max,25.800000,7.430000,85.000000


In [24]:
recipes[['cost','prep_time','cook_time','total_time','servings']].corr()

,cost,prep_time,cook_time,total_time,servings
cost,1.000000,0.215151,0.013010,0.114373,0.441473
prep_time,0.215151,1.000000,0.030761,0.505957,0.155287
cook_time,0.013010,0.030761,1.000000,0.877714,0.047306
total_time,0.114373,0.505957,0.877714,1.000000,0.115270
servings,0.441473,0.155287,0.047306,0.115270,1.000000


In [37]:
quick_recipes = recipes[recipes['total_time'] <= 30]

In [38]:
quick_recipes

,title,cost,cost_per_serving,servings,prep_time,cook_time,total_time
7,15 Bean Soup,6.76,0.85,8,0,0,0
11,Taco Pasta,8.64,1.44,6,5,25,30
15,Dan Dan Noodles,8.86,2.22,4,10,20,30
18,Marry Me White Bean Skillet,5.61,1.40,4,5,20,25
23,Seafood Lasagna,15.29,NaN,9,5,15,20
...,...,...,...,...,...,...,...
347,Broccoli Cheddar Stuffed Baked Potatoes,6.76,1.69,4,10,0,10
348,White Beans with Mushrooms and Marinara,6.54,1.64,4,5,25,30
351,Crunchy Chicken Ramen Stir Fry,7.10,1.78,4,10,15,25
353,BBQ Bean Sliders,5.85,1.46,4,15,15,30


In [39]:
cheap_recipes = recipes[recipes['cost_per_serving'] <= 2]

In [40]:
cheap_recipes

,title,cost,cost_per_serving,servings,prep_time,cook_time,total_time
0,Sliders,11.44,1.91,6,10,25,35
1,Ham Tetrazzini,7.65,1.28,6,15,45,60
2,Chicken Broth Bowl,7.53,1.89,4,15,35,50
4,Eggplant Curry,7.32,1.83,4,15,45,60
6,Slow Cooker Jambalaya,11.02,1.37,8,15,30,45
...,...,...,...,...,...,...,...
352,Roasted Broccoli Pasta with Lemon and Feta,3.69,0.92,4,15,25,40
353,BBQ Bean Sliders,5.85,1.46,4,15,15,30
354,Smoky Roasted Sausage and Vegetables,7.87,1.97,4,15,40,55
355,One Pot Spinach and Artichoke Pasta,9.07,1.62,6,10,15,25


In [52]:
quick_recipes[['cost', 'cost_per_serving', 'total_time']].describe()

,cost,cost_per_serving,total_time
count,146.000000,143.000000,146.000000
mean,8.865890,1.836923,23.109589
std,4.049155,0.720923,7.134301
min,2.920000,0.460000,0.000000
25%,6.317500,1.285000,20.000000
50%,8.010000,1.780000,25.000000
75%,10.850000,2.235000,30.000000
max,23.860000,4.280000,30.000000


In [51]:
cheap_recipes[['cost', 'cost_per_serving', 'total_time']].describe()

,cost,cost_per_serving,total_time
count,215.000000,215.000000,215.000000
mean,7.829767,1.434930,38.479070
std,2.988104,0.378857,16.308884
min,1.510000,0.270000,0.000000
25%,5.970000,1.165000,25.000000
50%,7.440000,1.460000,35.000000
75%,9.245000,1.780000,50.000000
max,18.430000,2.000000,85.000000


In [29]:
chicken_recipes = recipes[recipes['title'].str.contains('chicken', case=False)]

In [30]:
chicken_recipes

,title,cost,cost_per_serving,servings,prep_time,cook_time,total_time
2,Chicken Broth Bowl,7.53,1.89,4,15,35,50
5,Marry Me Chicken Meatballs,10.81,2.70,4,15,35,50
8,Thai Chicken Satay with Peanut Sauce,7.17,1.79,4,45,30,75
10,Chicken and Rice Casserole,7.60,1.27,6,15,55,70
12,Chicken Cobbler,9.05,1.81,5,10,40,50
...,...,...,...,...,...,...,...
337,Quick BBQ Chicken,4.98,1.25,4,10,15,25
338,Green Chile Chicken Enchiladas,8.46,1.41,6,25,45,70
342,Chicken Parmesan Meatballs,7.57,1.89,4,15,20,35
350,Creamy Chicken and Rice Skillet,7.12,1.78,4,5,45,50


In [31]:
beef_recipes = recipes[recipes['title'].str.contains('beef', case=False)]

In [32]:
beef_recipes

,title,cost,cost_per_serving,servings,prep_time,cook_time,total_time
55,Beef Enchiladas,7.36,1.23,6,10,50,60
103,Corned Beef and Cabbage,22.63,3.77,6,10,0,10
116,Instant Pot Beef Stew,15.87,2.65,6,15,0,15
123,Crockpot Beef Stew,14.08,1.41,8,15,15,30
129,Beef Stroganoff,5.54,1.39,4,10,20,30
133,Beef Stew,16.53,2.75,6,15,50,65
146,Stuffed Shells With Ground Beef,13.63,2.27,6,15,45,60
222,Beef And Tomato Rice Bowl,7.78,1.95,4,5,30,35
264,Sautéed Beef Cabbage and Rice,5.90,1.48,4,10,25,35
293,Curried Ground Beef with Peas and Potatoes,6.99,1.17,6,10,25,35


In [33]:
one_pot_recipes = recipes[recipes['title'].str.contains('one pot', case=False)]

In [34]:
one_pot_recipes = recipes[recipes['title'].str.contains('one pot|one-pan|one-pot', case=False)]

In [35]:
one_pot_recipes

,title,cost,cost_per_serving,servings,prep_time,cook_time,total_time
127,One Pot Chicken and Rice,6.18,1.54,4,10,40,50
130,One Pot Creamy Pesto Chicken Pasta,11.77,2.94,4,5,20,25
188,One Pot Chili Mac,8.98,1.50,6,5,30,35
257,One Pot Lemon Pepper Chicken with Orzo,7.54,1.89,4,10,35,45
288,One Pot Veggie Pasta,9.43,2.36,4,10,20,30
301,One Pot Cheeseburger Pasta,5.03,1.26,4,10,20,30
332,One Pot Lemon Artichoke Chicken and Rice,7.39,1.85,4,5,25,30
340,One Pot Creamy Sun Dried Tomato Pasta,4.22,1.06,4,10,20,30
355,One Pot Spinach and Artichoke Pasta,9.07,1.62,6,10,15,25


In [48]:
chicken_recipes[['cost', 'cost_per_serving', 'total_time']].describe()

,cost,cost_per_serving,total_time
count,107.000000,107.000000,107.000000
mean,9.310561,2.047196,38.420561
std,2.812559,0.731658,15.323577
min,3.460000,0.800000,5.000000
25%,7.340000,1.515000,25.000000
50%,9.020000,1.940000,35.000000
75%,11.090000,2.480000,50.000000
max,17.340000,4.300000,75.000000


In [49]:
beef_recipes[['cost', 'cost_per_serving', 'total_time']].describe()

,cost,cost_per_serving,total_time
count,11.000000,11.000000,11.000000
mean,11.277273,2.000909,36.363636
std,5.579065,0.804319,18.178409
min,5.540000,1.170000,10.000000
25%,7.175000,1.400000,27.500000
50%,7.780000,1.940000,35.000000
75%,14.975000,2.460000,47.500000
max,22.630000,3.770000,65.000000


In [50]:
one_pot_recipes[['cost', 'cost_per_serving', 'total_time']].describe()

,cost,cost_per_serving,total_time
count,9.000000,9.000000,9.000000
mean,7.734444,1.780000,33.333333
std,2.364683,0.575478,8.660254
min,4.220000,1.060000,25.000000
25%,6.180000,1.500000,30.000000
50%,7.540000,1.620000,30.000000
75%,9.070000,1.890000,35.000000
max,11.770000,2.940000,50.000000
